# Test BNF API — POC Data

Objectif :

- tester BNF API sur une première liste de 20 EAN/ISBN ;
- vérifier la couverture de l’API ;
- analyser rapidement la complétude des métadonnées ;
- capturer les erreurs sans bloquer le notebook ;
- exporter les résultats pour analyse future.

## Import

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

from collectionlens.api.bnf_api import search_book_by_isbn
from collectionlens.benchmark.api_benchmark import build_source_table

In [2]:
import requests
isbn = "9782351423554"
params = {
    "version": "1.2",
    "operation": "searchRetrieve",
    "query": f'bib.isbn="{isbn}"',
    "recordSchema": "dublincore",
    "maximumRecords": 1,
}

r = requests.get("https://catalogue.bnf.fr/api/SRU", params=params, timeout=10)

print(r.status_code)
print(r.url)
print(r.text[:2000])

200
https://catalogue.bnf.fr/api/SRU?version=1.2&operation=searchRetrieve&query=bib.isbn%3D%229782351423554%22&recordSchema=dublincore&maximumRecords=1
<?xml version="1.0" encoding="UTF-8"?><srw:searchRetrieveResponse xmlns:srw="http://www.loc.gov/zing/srw/" xmlns="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:ixm="http://catalogue.bnf.fr/namespaces/InterXMarc" xmlns:mn="http://catalogue.bnf.fr/namespaces/motsnotices" xmlns:sd="http://www.loc.gov/zing/srw/diagnostic/">
<srw:version>1.2</srw:version>
<srw:echoedSearchRetrieveRequest>
<srw:version>1.2</srw:version>
<srw:query>bib.isbn="9782351423554"</srw:query>
</srw:echoedSearchRetrieveRequest>
<srw:numberOfRecords>1</srw:numberOfRecords>
<srw:records>
<srw:record>
<srw:recordSchema>dc</srw:recordSchema>
<srw:recordPacking>xml</srw:recordPacking>
<srw:recordData>
<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" 

## Liste ISBN test

In [3]:
isbns = [
    "9782351423554",
    "9791042019396",
    "9782820354488",
    "9782290042298",
    "9782845808331",
    "9782413042341",
    "9782384967421",
    "9791043301087",
    "9782374123035",
    "9782374126173",
    "9791039143431",
    "9791026854920",
    "9791039140652",
    "9791026856283",
    "9791039147156",
    "9782203001190",
    "9782858150045",
    "9791038206229",
    "9782822244787",
    "9791038209657",
]

## Exécution du test

In [4]:
df_bnf = build_source_table(
    source_name="bnf",
    search_func=search_book_by_isbn,
    isbns=isbns,
)

## Analyse rapide

In [5]:

coverage_rate = df_bnf["found"].mean() * 100

print(f"Taux de couverture BNF api : {coverage_rate:.2f}%")

Taux de couverture BNF api : 45.00%


In [6]:

df_bnf["found"].value_counts(dropna=False)

found
False    11
True      9
Name: count, dtype: int64

## Analyse des erreurs

In [7]:
df_bnf.loc[~df_bnf["found"], ["isbn_query", "error"]]

,isbn_query,error
1,9791042019396,no_result
3,9782290042298,no_result
6,9782384967421,no_result
7,9791043301087,no_result
8,9782374123035,no_result
9,9782374126173,no_result
11,9791026854920,no_result
13,9791026856283,no_result
14,9791039147156,no_result
15,9782203001190,no_result


## Complétude des métadonnées

In [8]:

metadata_fields = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "language",
    "description",
    "categories",
    "thumbnail",
]

available_fields = [
    field for field in metadata_fields
    if field in df_bnf.columns
]

metadata_completeness = (
    df_bnf[available_fields]
    .notna()
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

metadata_completeness.columns = ["field", "completion_rate"]

metadata_completeness

,field,completion_rate
0,title,45.0
1,authors,45.0
2,publisher,45.0
3,published_date,45.0
4,language,45.0
5,description,45.0
6,categories,45.0
7,thumbnail,0.0


In [9]:
df_bnf_ok = df_bnf[df_bnf["found"]].copy()
df_bnf_ko = df_bnf[~df_bnf["found"]].copy()

display(df_bnf_ok)
display(df_bnf_ko)

,source,isbn_query,found,error,status_code,source_id,isbn,title,subtitle,authors,...,average_rating,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,format,types,rights,raw_data
0,bnf,9782351423554,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb41404031j,9782351423554,Vinland saga. 1 / Makoto Yukimura,NaN,"[Yukimura, Makoto (1976-....). Auteur du texte]",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb41404031...,"1 vol. (213 p.) : ill. en noir et en coul., co...","[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
2,bnf,9782820354488,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb48763865j,9782820354488,Blue Exorcist. 32,NaN,"[Katō, Kazue (1980-....)]",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb48763865...,18 cm,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
4,bnf,9782845808331,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb410390294,9782845808331,"Dragon quest : la quête de Daï. 1 / scénario, ...",NaN,"[Sanjō, Riku (1964-....). Auteur du texte, Ina...",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb41039029...,"1 vol. (199 p.) : ill., couv. ill., couv. ill....","[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
5,bnf,9782413042341,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb47221792v,9782413042341,"Dragon quest, the adventure of Dai. 1, Les dis...",NaN,"[Sanjō, Riku (1964-....). Auteur du texte]",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb47221792...,1 vol. (326 p.) : ill. ; 18 cm,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
10,bnf,9791039143431,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb487470845,9791039143431,Star Wars : La citadelle hurlante,NaN,"[Aaron, Jason (1973-....), Gillen, Kieron (197...",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb48747084...,1 vol. (non paginé [272 p.]) ; 23 cm,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
12,bnf,9791039140652,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb48579742x,9791039140652,La Chose ([Éd. spéciale] exclusivité FNAC) [sc...,NaN,"[Johns, Geoff (1973-....). Auteur du texte, Wi...",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb48579742...,1 vol. (non paginé [ca 120] p.) : ill. en coul...,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
16,bnf,9782858150045,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb409749703,9782858150045,Titine au bistrot / Lindingre,NaN,"[Lindingre, Yan (1969-....). Auteur du texte]",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb40974970...,"1 vol. (46 p.) : ill. en coul., couv. ill. en ...","[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
17,bnf,9791038206229,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb47413282s,9791038206229,"Fleurs de pavés / scénario, dessin, couleurs, ...",NaN,"[Frécon, Sylvain (1972-....). Auteur du texte]",...,NaN,NaN,NaN,NaN,NaN,[http://catalogue.bnf.fr/ark:/12148/cb47413282...,1 vol. (54 p.) : ill. en coul. ; 30 cm,"[{'@xml:lang': 'fre', '#text': 'texte imprimé'...","[{'@xml:lang': 'fre', '#text': 'Catalogue en l...","{'srw:recordSchema': 'dc', 'srw:recordPacking'..."
18,bnf,9782822244787,True,None,200,http://catalogue.bnf.fr/ark:/12148/cb47588925q,9782822244787,"Stranger trucs / Antoine Piers, Novy ; [d

,source,isbn_query,found,error,status_code,source_id,isbn,title,subtitle,authors,...,average_rating,ratings_count,preview_link,info_link,canonical_volume_link,industry_identifiers,format,types,rights,raw_data
1,bnf,9791042019396,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,bnf,9782290042298,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,bnf,9782384967421,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,bnf,9791043301087,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,bnf,9782374123035,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,bnf,9782374126173,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,bnf,9791026854920,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
13,bnf,9791026856283,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,bnf,9791039147156,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,bnf,9782203001190,False,no_result,200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:

output_dir = PROJECT_ROOT / "data" / "processed" / "bnf"
output_dir.mkdir(parents=True, exist_ok=True)

full_output_path = output_dir / "bnf_results_full.csv"
ok_output_path = output_dir / "bnf_results_ok.csv"
ko_output_path = output_dir / "bnf_results_ko.csv"

df_bnf.to_csv(full_output_path, index=False)
df_bnf_ok.to_csv(ok_output_path, index=False)
df_bnf_ko.to_csv(ko_output_path, index=False)

# Premiers constats

Les premiers tests réalisés sur l’API BNF montrent des résultats encourageants dans le cadre du POC CollectionLens.

## Couverture des ouvrages

Le taux de couverture observé sur ce premier mini-dataset est d’environ 45 %.

Ce résultat est nettement supérieur aux premiers essais précédents et rapproche la BNF des performances observées avec Google Books sur cet échantillon.

Plusieurs références françaises pertinentes ont été retrouvées, notamment :
- mangas ;
- bandes dessinées ;
- éditions françaises récentes.

## Complétude des métadonnées

La BNF fournit plusieurs métadonnées intéressantes avec un taux de remplissage relativement homogène :

- titre : 45 % ;
- auteurs : 45 % ;
- éditeur : 45 % ;
- date de publication : 45 % ;
- langue : 45 % ;
- description : 45 %.

La présence des descriptions constitue un point particulièrement intéressant pour les futurs usages de recommandation et de RAG.

En revanche, certaines métadonnées restent absentes ou limitées :
- thumbnails ;
- catégories structurées ;
- informations enrichies de séries/volumes.

## Limites identifiées

Les principales limites observées sont les suivantes :

- structure XML/SRU plus complexe que les APIs JSON classiques ;
- normalisation des notices encore partielle ;
- extraction métier à améliorer ;
- absence d’images de couverture ;
- hétérogénéité de certaines notices.

## Conclusion POC

La BNF apparaît désormais comme une source très prometteuse pour améliorer la couverture des éditions françaises dans CollectionLens.

Les résultats obtenus justifient :
- la poursuite des benchmarks sur des volumes plus importants ;
- l’amélioration de la normalisation BNF ;
- l’intégration future dans une stratégie multi-sources ;
- l’évaluation de son intérêt pour les usages de recommandation et de RAG.